# Les 01 - Introductie tot AI-agenten

Welkom bij de eerste les van de cursus **AI-agenten voor beginners**!

Een **AI-agent** is een programma dat een groot taalmodel (LLM) gebruikt als zijn redeneermotor en *acties* kan uitvoeren in de echte wereld — API's aanroepen, databases raadplegen of code uitvoeren — om namens een gebruiker een doel te bereiken.

In dit notitieboek bouw je je eerste agent: een **reisagent** die vakantiebestemmingen aanbeveelt. Onderweg leer je hoe je:

1. Verbinding maakt met de Microsoft Foundry Agent Service met behulp van het **Microsoft Agent Framework**.
2. De agent een **tool** geeft — een gewone Python-functie die hij kan aanroepen.
3. De agent uitvoert en zijn reactie inspecteert.
4. De reactie van de agent token-voor-token streamt.


## Installatie

Voordat je deze notebook uitvoert, zorg ervoor dat je:

1. **Een Microsoft Foundry-project** hebt met een geïmplementeerd chatmodel (bijv. `gpt-4.1-mini`).
2. **Ingelogd bent met de Azure CLI** — voer `az login` uit in je terminal.
3. **De vereiste omgevingsvariabelen hebt ingesteld:**
   - `AZURE_AI_PROJECT_ENDPOINT` — je Microsoft Foundry-projectendpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — de naam van je geïmplementeerde model.

De onderstaande cel installeert de Python-pakketten die je nodig hebt.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Je Eerste Agent Maken

Een agent heeft twee dingen nodig:

- **Instructies** die vertellen *wie hij is* en *hoe hij zich moet gedragen* (een systeemprompt).
- **Gereedschappen** — Python-functies aangeduid met `@tool` die de agent kan aanroepen om informatie op te halen of acties uit te voeren.

Hieronder definiëren we een eenvoudig gereedschap dat een lijst met populaire vakantiebestemmingen retourneert. De agent zal dit gereedschap gebruiken wanneer een gebruiker om reisaanbevelingen vraagt.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Antwoorden

Voor een meer interactieve ervaring kun je de reactie van de agent **streamen**. In plaats van te wachten op het volledige antwoord, levert de agent tekstfragmenten terwijl ze worden gegenereerd. Dit is vooral handig in chatinterfaces waar je de output in realtime wilt tonen.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Samenvatting

In deze les heb je geleerd hoe je:

- **Een provider maakt** die verbinding maakt met de Microsoft Foundry Agent Service via `FoundryChatClient`.
- **Een tool definieert** met de `@tool` decorator zodat de agent je Python-functies kan aanroepen.
- **De agent uitvoert** met een gebruikersbericht en de reactie afdrukt.
- **Reacties streamt** voor realtime output.

In de volgende les zullen we agentgebaseerde frameworks dieper onderzoeken en leren hoe we agenten krachtigere tools en meervoudige stappen redeneringsmogelijkheden kunnen geven.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
